In [1]:
# ==============================================================================
# 🚀 MÁSTER UNIVERSITARIO EN CIENCIA DE DATOS MASIVOS
# Course: Generative Artificial Intelligence - Project (Option 2: RAG System)
# Topic: Secure Entity Resolution Pipeline using Gemini 1.5 PRO & HuggingFace
# ==============================================================================

# ------------------------------------------------------------------------------
# STEP 0: INSTALL DEPENDENCIES
# ------------------------------------------------------------------------------
!pip install -qU langchain-core langchain-chroma langchain-google-genai langchain-huggingface sentence-transformers

import os
import getpass
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser

# ------------------------------------------------------------------------------
# STEP 1: SECURE API KEY MANAGEMENT
# ------------------------------------------------------------------------------
if "GOOGLE_API_KEY" not in os.environ:
    print("Please enter your Google Gemini API Key below:")
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key: ")

# ------------------------------------------------------------------------------
# STEP 2: EXPLORATORY DATA ANALYSIS (EDA) & DATA CLEANING
# ------------------------------------------------------------------------------
print("\n--- Phase 1: Exploratory Data Analysis & Cleaning ---")

# Load the dataset
df = pd.read_csv('Capital_Calls_DB.csv')
df.fillna("Unknown", inplace=True)

# Guardrail: Correct human data-entry mistakes
df.loc[df['S&P'] == 'Aaa', 'S&P'] = 'NR'
df.loc[df["Moody's"] == 'AAA', "Moody's"] = 'NR'

print(f"Dataset loaded successfully with {len(df)} records.")
print("Previewing the structured data:")
print(df.head())

# Converting Pandas DataFrame directly to LangChain Core Documents
documents = []
for _, row in df.iterrows():
    doc = Document(
        page_content=str(row['LP Name']),
        metadata={
            "Investor Type": str(row['Investor Type']),
            "Country": str(row['Country']),
            "S&P": str(row['S&P']),
            "Moody's": str(row["Moody's"])
        }
    )
    documents.append(doc)

print(f"Total core documents structured for Vector Store: {len(documents)}")

# ------------------------------------------------------------------------------
# STEP 3: VECTOR DB SETUP & RETRIEVAL PIPELINE (LOCAL EMBEDDINGS)
# ------------------------------------------------------------------------------
print("\n--- Phase 2: Building Vector DB & Retriever ---")

# Using local HuggingFace embeddings (Free and highly efficient)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="lp_entities"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

test_query = "Border to Coast"
retrieved_docs = retriever.invoke(test_query)
print(f"Sample retrieval test for '{test_query}' found {len(retrieved_docs)} candidates.")

# ------------------------------------------------------------------------------
# STEP 4: GENERATION PIPELINE & SYSTEM GUARDRAILS
# UPGRADED TO: gemini-1.5-pro for advanced reasoning and strict guardrail adherence
# ------------------------------------------------------------------------------
print("\n--- Phase 3: Constructing RAG Chain with Gemini 1.5 PRO ---")

llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.0)

template = """
You are an expert Entity Resolution AI engine operating within an Investment Banking context.
Your task is to analyze the user-provided "Input LP Name" against a set of retrieved "Candidate LP List" pulled from our database.

Retrieved Candidate LP List (Context):
{context}

Input LP Name to Resolve (Query):
{query}

CRITICAL RULES & GUARDRAILS:
1. Pay extreme attention to Series/Class variations (e.g., "Fund A" vs "Fund B", "Series I" vs "Series II"). These are legally separate corporate entities. If a variance in series/class is detected, you MUST reject the match.
2. Minor typos or punctuation differences in corporate suffixes (e.g., "LLC" vs "L.L.C.", "LP" vs "L.P.") are acceptable for a successful match if the core name is identical.
3. Your output must strictly be a valid JSON object. Do not include markdown formatting like ```json or any conversational text.

OUTPUT FORMAT (JSON):
{{
  "matched_candidate": "The exact matched LP Name from the candidate list, or 'NONE' if no valid match exists",
  "confidence_score": A float value between 0.00 and 1.00 indicating match security,
  "explanation": "A step-by-step analytical justification of your decision referencing the rules",
  "trigger_human_review": true or false (Must be true if confidence_score is less than 0.85)
}}
"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n".join([f"Name: {doc.page_content} | Info: {doc.metadata}" for doc in docs])

parser = JsonOutputParser()

rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

# ------------------------------------------------------------------------------
# STEP 5: PIPELINE EVALUATION & SCENARIOS
# ------------------------------------------------------------------------------
print("\n--- Phase 4: Executing Evaluation Scenarios ---")

query_1 = "Border to Coast Bedfordshire L.P."
print(f"\n[Test 1] Input: '{query_1}'")
result_1 = rag_chain.invoke(query_1)
print("Output JSON:")
print(result_1)

query_2 = "Border to Coast Bedfordshire LP Series II"
print(f"\n[Test 2] Input: '{query_2}'")
result_2 = rag_chain.invoke(query_2)
print("Output JSON:")
print(result_2)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

FileNotFoundError: [Errno 2] No such file or directory: 'Capital_Calls_DB.csv'